# Étude de la quantification d'incertitude par Monte Carlo Dropout

**Contexte** : Dans le TP4, le modèle U-Net entraîné sur le dataset d'endoscopie (100 paires image/masque 256×256) a obtenu :
- Dice = 0 à 10 epochs (le modèle prédit uniquement le fond majoritaire)
- Dice ≈ 0.47 à 50 epochs (modèle fragile, surtout sur les petites régions)

**Hypothèse** : En exploitant les couches Dropout déjà présentes dans le U-Net (0.1→0.3), le Monte Carlo Dropout permet d'identifier les prédictions peu fiables sans ré-entraînement.

**Plan** :
1. Setup
2. Expérience A — Sensibilité à N (nombre de passes)
3. Expérience B — Corrélation incertitude ↔ erreur
4. Expérience C — Rejet sélectif
5. Expérience D — Galerie de cas
6. Discussion

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import time
import cv2
from pathlib import Path
from scipy.stats import pearsonr
from tensorflow.keras.models import load_model

from app.services.uncertainty import mc_dropout_predict, colorize
from app.services.metrics import dice, iou

plt.rcParams['font.size'] = 10
plt.rcParams['figure.dpi'] = 150

## 1. Setup

Chargement du modèle U-Net entraîné et des images de validation.

In [ ]:
MODEL_PATH = "../app/models_registry/unet_endoscopy.hdf5"
model = load_model(MODEL_PATH, compile=False)
print(f"Modèle chargé : {model.count_params()} paramètres")
print(f"Couches Dropout : {[l.name for l in model.layers if 'dropout' in l.name.lower()]}")

In [ ]:
# Charger les images de validation
# Adapter le chemin vers votre dataset
DATA_DIR = Path("/path/to/endoscopy_dataset")
IMG_SIZE = 256

images_dir = DATA_DIR / "images"
masks_dir = DATA_DIR / "masks"

image_paths = sorted(images_dir.glob("*.png")) + sorted(images_dir.glob("*.jpg"))

# Prendre les 10 dernières comme validation
val_paths = image_paths[-10:]

val_images = []
val_masks = []

for ip in val_paths:
    img = cv2.imread(str(ip), cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    val_images.append(img)
    
    mp = masks_dir / ip.name
    if not mp.exists():
        mp = masks_dir / ip.with_suffix(".png").name
    msk = cv2.imread(str(mp), cv2.IMREAD_GRAYSCALE)
    msk = cv2.resize(msk, (IMG_SIZE, IMG_SIZE))
    val_masks.append((msk > 127).astype(np.uint8) * 255)

print(f"{len(val_images)} images de validation chargées")

## 2. Expérience A — Sensibilité à N

Pour N ∈ {5, 10, 20, 30, 50}, on mesure :
- Dice moyen sur le set de validation
- Confiance moyenne
- Temps d'inférence moyen

Objectif : identifier le plateau au-delà duquel augmenter N n'apporte plus de bénéfice.

In [ ]:
N_VALUES = [5, 10, 20, 30, 50]

results_a = {n: {"dices": [], "confs": [], "times": []} for n in N_VALUES}

for n in N_VALUES:
    print(f"\nN = {n}")
    for i, (img, gt) in enumerate(zip(val_images, val_masks)):
        t0 = time.perf_counter()
        mask, std_map, entropy_map, conf = mc_dropout_predict(model, img, n_passes=n)
        elapsed = time.perf_counter() - t0
        
        d = dice(gt, mask)
        results_a[n]["dices"].append(d)
        results_a[n]["confs"].append(conf)
        results_a[n]["times"].append(elapsed)
    
    mean_dice = np.mean(results_a[n]["dices"])
    mean_conf = np.mean(results_a[n]["confs"])
    mean_time = np.mean(results_a[n]["times"])
    print(f"  Dice moyen: {mean_dice:.4f}")
    print(f"  Confiance moyenne: {mean_conf:.4f}")
    print(f"  Temps moyen: {mean_time:.2f}s")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

mean_dices = [np.mean(results_a[n]["dices"]) for n in N_VALUES]
mean_confs = [np.mean(results_a[n]["confs"]) for n in N_VALUES]
mean_times = [np.mean(results_a[n]["times"]) for n in N_VALUES]

axes[0].plot(N_VALUES, mean_dices, "o-", color="#1e40af", linewidth=1.5)
axes[0].set_xlabel("N (passes)")
axes[0].set_ylabel("Dice moyen")
axes[0].set_title("Dice vs N")
axes[0].grid(True, alpha=0.3)

axes[1].plot(N_VALUES, mean_confs, "o-", color="#16a34a", linewidth=1.5)
axes[1].set_xlabel("N (passes)")
axes[1].set_ylabel("Confiance moyenne")
axes[1].set_title("Confiance vs N")
axes[1].grid(True, alpha=0.3)

axes[2].plot(N_VALUES, mean_times, "o-", color="#dc2626", linewidth=1.5)
axes[2].set_xlabel("N (passes)")
axes[2].set_ylabel("Temps (s)")
axes[2].set_title("Temps d'inférence vs N")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../../docs/exp_a_sensitivity_N.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Expérience B — Corrélation incertitude ↔ erreur

Pour chaque image i, on calcule :
- `mean_std_i` : incertitude moyenne (écart-type moyen sur les pixels)
- `error_i = 1 - Dice_i` : erreur de segmentation

Si la corrélation de Pearson r > 0.5 avec p < 0.05, le modèle « sait quand il ne sait pas ».

In [ ]:
N_REF = 30
uncertainties = []
errors = []
dices_b = []
confs_b = []
std_maps_b = []

for i, (img, gt) in enumerate(zip(val_images, val_masks)):
    mask, std_map, entropy_map, conf = mc_dropout_predict(model, img, n_passes=N_REF)
    
    d = dice(gt, mask)
    mean_std = float(np.mean(std_map))
    
    uncertainties.append(mean_std)
    errors.append(1.0 - d)
    dices_b.append(d)
    confs_b.append(conf)
    std_maps_b.append(std_map)
    
    print(f"Image {i}: Dice={d:.4f}, Incertitude={mean_std:.4f}, Confiance={conf:.4f}")

r, p_value = pearsonr(uncertainties, errors)
print(f"\nCorrélation de Pearson : r = {r:.4f}, p = {p_value:.6f}")

if r > 0.5 and p_value < 0.05:
    print("→ Le modèle SAIT quand il ne sait pas (corrélation significative).")
else:
    print("→ Corrélation non significative ou faible.")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(uncertainties, errors, c="#1e40af", s=40, edgecolors="white", linewidths=0.5)

# Droite de régression
z = np.polyfit(uncertainties, errors, 1)
p = np.poly1d(z)
x_line = np.linspace(min(uncertainties), max(uncertainties), 50)
ax.plot(x_line, p(x_line), "--", color="#dc2626", linewidth=1, alpha=0.7)

ax.set_xlabel("Incertitude moyenne (σ)")
ax.set_ylabel("Erreur (1 - Dice)")
ax.set_title(f"Corrélation incertitude ↔ erreur\nr={r:.3f}, p={p_value:.4f}")
ax.grid(True, alpha=0.3)

for i in range(len(uncertainties)):
    ax.annotate(f"{i}", (uncertainties[i], errors[i]), fontsize=7, ha="center", va="bottom")

plt.tight_layout()
plt.savefig("../../docs/exp_b_correlation.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Expérience C — Rejet sélectif

On trie les images par confiance décroissante et on rejette les τ% les moins confiantes.
Pour τ ∈ {0, 10, 20, 30, 40, 50}%, on calcule le Dice moyen sur les images restantes.

Message attendu : « Rejeter les 20% les moins confiants fait passer le Dice de X à Y. »

In [ ]:
# Trier par confiance décroissante
indices_sorted = np.argsort(confs_b)[::-1]
dices_sorted = np.array(dices_b)[indices_sorted]
confs_sorted = np.array(confs_b)[indices_sorted]

tau_values = [0, 10, 20, 30, 40, 50]
rejection_results = []

n_total = len(dices_sorted)
dice_all = np.mean(dices_sorted)

print(f"{'τ (%)':>8} {'N retenues':>12} {'Dice moyen':>12} {'Gain':>10}")
print("-" * 45)

for tau in tau_values:
    n_keep = max(1, int(n_total * (100 - tau) / 100))
    dice_kept = np.mean(dices_sorted[:n_keep])
    gain = (dice_kept - dice_all) / dice_all * 100
    rejection_results.append((tau, n_keep, dice_kept, gain))
    print(f"{tau:>8} {n_keep:>12} {dice_kept:>12.4f} {gain:>+9.1f}%")

# Message résumé
dice_0 = rejection_results[0][2]
dice_20 = [r for r in rejection_results if r[0] == 20][0][2]
gain_20 = (dice_20 - dice_0) / dice_0 * 100
print(f"\n→ Rejeter les 20% les moins confiants fait passer le Dice de {dice_0:.4f} à {dice_20:.4f} (+{gain_20:.1f}%)")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
taus = [r[0] for r in rejection_results]
dice_means = [r[2] for r in rejection_results]

ax.plot(taus, dice_means, "o-", color="#1e40af", linewidth=1.5)
ax.fill_between(taus, dice_means[0], dice_means, alpha=0.1, color="#1e40af")
ax.set_xlabel("Taux de rejet τ (%)")
ax.set_ylabel("Dice moyen")
ax.set_title("Rejet sélectif : Dice vs taux de rejet")
ax.grid(True, alpha=0.3)
ax.axhline(y=dice_means[0], color="#6b6b6b", linestyle="--", linewidth=0.8, alpha=0.5)

plt.tight_layout()
plt.savefig("../../docs/exp_c_rejection.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Expérience D — Galerie de cas

On sélectionne 3 cas représentatifs :
- **Cas 1** : Haute confiance + bon Dice → le modèle fonctionne bien
- **Cas 2** : Haute confiance + mauvais Dice → danger (confiance mal calibrée)
- **Cas 3** : Basse confiance + mauvais Dice → le modèle est honnête sur ses limites

Pour chaque cas : image | GT | prédiction | carte σ

In [ ]:
# Identifier les cas
dices_arr = np.array(dices_b)
confs_arr = np.array(confs_b)

median_dice = np.median(dices_arr)
median_conf = np.median(confs_arr)

# Cas 1 : haute confiance + bon Dice
good_mask = (dices_arr > median_dice) & (confs_arr > median_conf)
if good_mask.any():
    case1_idx = np.where(good_mask)[0][np.argmax(dices_arr[good_mask])]
else:
    case1_idx = np.argmax(dices_arr)

# Cas 3 : basse confiance + mauvais Dice
bad_mask = (dices_arr <= median_dice) & (confs_arr <= median_conf)
if bad_mask.any():
    case3_idx = np.where(bad_mask)[0][np.argmin(dices_arr[bad_mask])]
else:
    case3_idx = np.argmin(dices_arr)

# Cas 2 : haute confiance + mauvais Dice (si existant)
danger_mask = (dices_arr <= median_dice) & (confs_arr > median_conf)
if danger_mask.any():
    case2_idx = np.where(danger_mask)[0][np.argmin(dices_arr[danger_mask])]
else:
    # Prendre un cas intermédiaire
    remaining = set(range(len(dices_arr))) - {case1_idx, case3_idx}
    case2_idx = list(remaining)[0] if remaining else 0

cases = [
    (case1_idx, "Haute confiance, bon Dice"),
    (case2_idx, "Haute confiance, mauvais Dice (danger)"),
    (case3_idx, "Basse confiance, mauvais Dice (honnête)"),
]

for idx, label in cases:
    print(f"{label}: image {idx}, Dice={dices_b[idx]:.4f}, Confiance={confs_b[idx]:.4f}")

In [ ]:
fig = plt.figure(figsize=(12, 9))
gs = gridspec.GridSpec(3, 4, hspace=0.35, wspace=0.15)

col_titles = ["Image", "Ground Truth", "Prédiction", "Carte σ"]

for row, (idx, label) in enumerate(cases):
    img = val_images[idx]
    gt = val_masks[idx]
    mask_pred, std_map, _, conf = mc_dropout_predict(model, img, n_passes=N_REF)
    std_colored = colorize(std_map, "inferno")
    
    panels = [img, gt, mask_pred, std_colored]
    cmaps = ["gray", "gray", "gray", None]
    
    for col in range(4):
        ax = fig.add_subplot(gs[row, col])
        if cmaps[col]:
            ax.imshow(panels[col], cmap=cmaps[col])
        else:
            ax.imshow(panels[col])
        ax.axis("off")
        
        if row == 0:
            ax.set_title(col_titles[col], fontsize=10, fontweight="medium")
    
    # Label à gauche
    ax0 = fig.add_subplot(gs[row, 0])
    ax0.set_ylabel(f"{label}\nDice={dices_b[idx]:.3f} Conf={confs_b[idx]:.3f}",
                   fontsize=8, rotation=0, labelpad=80, va="center")

plt.savefig("../../docs/uncertainty_showcase.png", dpi=300, bbox_inches="tight")
print("Figure sauvegardée dans docs/uncertainty_showcase.png")
plt.show()

## 6. Discussion

### Lien avec le TP4

Le TP4 a montré que le U-Net entraîné sur seulement 100 images obtient un Dice de ~0.47, avec des performances très faibles sur les petites régions (le modèle favorise le fond majoritaire). 

**Question** : L'incertitude est-elle plus élevée sur les petites régions non détectées ?
→ Vérifier en comparant la carte σ avec les régions de petite taille dans le GT.

### Lien avec le TP2

**Question** : La std explose-t-elle sur les reflets spéculaires endoscopiques ?
→ Les reflets spéculaires créent des artéfacts d'intensité élevée qui perturbent la segmentation. Le MC Dropout devrait produire une incertitude élevée dans ces zones.

### Limites

- **Seulement 10 images de validation** : les résultats statistiques (Pearson, rejet sélectif) sont indicatifs mais pas robustes. Comme noté dans le TP4, l'absence de test set indépendant limite la généralisabilité.
- **Incertitude épistémique uniquement** : le MC Dropout capture l'incertitude liée au modèle (manque de données), pas l'incertitude aléatorique (bruit inhérent aux images).
- **Temps d'inférence** : N=30 passes multiplie le temps par ~30. Sur CPU, cela peut dépasser 10 secondes par image.

### Perspective

Combiner MC Dropout avec la perte Dice+BCE (déjà implémentée dans la version améliorée) pour obtenir à la fois :
- Une meilleure performance de segmentation (Dice > 0.75)
- Une meilleure calibration de l'incertitude

L'incertitude pourrait également servir de critère pour l'apprentissage actif : sélectionner les images les plus incertaines pour annotation et ré-entraînement.

In [ ]:
# Résumé des résultats
print("=" * 50)
print("RÉSUMÉ DES RÉSULTATS")
print("=" * 50)
print(f"\nDice moyen (N={N_REF}): {np.mean(dices_b):.4f}")
print(f"Confiance moyenne: {np.mean(confs_b):.4f}")
print(f"Corrélation Pearson incertitude↔erreur: r={r:.4f}, p={p_value:.6f}")
print(f"\nRejet sélectif (20%): Dice {dice_0:.4f} → {dice_20:.4f} (+{gain_20:.1f}%)")
print(f"\nCes valeurs sont à reporter dans le README et le rapport.")